# Football Club Financial Health Analysis (CSV Version)
## Accounting Data Product Assignment - Track 4

**Student Name:** Hongyu.Pan
**Date:** April 2026

### Objective
Analyze the financial health of top European football clubs with focus on cost control and revenue diversification.

### Data Files (CSV format)
- **Financial Data**: `financial_data.csv` (converted from Excel)
- **Performance Data**: `performance_data.csv` (converted from Excel)

### Note
This version uses CSV files for better compatibility and avoids Excel encoding issues.

## Step 1: Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

print("✓ Libraries imported")

In [ ]:
# Load CSV files with UTF-8 encoding
print("Loading financial_data.csv...")
financial_df = pd.read_csv('financial_data.csv', encoding='utf-8-sig')
print(f"✓ Loaded: {financial_df.shape[0]} rows, {financial_df.shape[1]} columns")
print("Columns:", list(financial_df.columns))
print("\nFirst 3 rows:")
print(financial_df.head(3))

In [ ]:
print("\nLoading performance_data.csv...")
performance_df = pd.read_csv('performance_data.csv', encoding='utf-8-sig')
print(f"✓ Loaded: {performance_df.shape[0]} rows, {performance_df.shape[1]} columns")
print("Columns:", list(performance_df.columns))
print("\nFirst 3 rows:")
print(performance_df.head(3))

## Step 2: Data Transformation

Calculate required metrics:

In [ ]:
# Create working copy
df = financial_df.copy()

# Rename columns for easier access
# Note: CSV column names match exactly what we need
df = df.rename(columns={
    'Club Name': 'Club',
    'Total Revenue (€m)': 'Total_Revenue',
    'Commercial Revenue (€m)': 'Commercial_Revenue',
    'Wages/Revenue Ratio (%)': 'Wages_Ratio'
})

# Convert numeric columns to proper types, coercing errors to NaN
df['Total_Revenue'] = pd.to_numeric(df['Total_Revenue'], errors='coerce')
df['Commercial_Revenue'] = pd.to_numeric(df['Commercial_Revenue'], errors='coerce')
df['Wages_Ratio'] = pd.to_numeric(df['Wages_Ratio'], errors='coerce')

# Calculate new columns
df['Wages_Amount'] = df['Total_Revenue'] * (df['Wages_Ratio'] / 100)
df['Commercial_Share'] = (df['Commercial_Revenue'] / df['Total_Revenue']) * 100

# Display transformed data
print("=== TRANSFORMED FINANCIAL DATA ===")
print(df[['Club', 'Year', 'Total_Revenue', 'Commercial_Revenue', 'Wages_Ratio', 'Wages_Amount', 'Commercial_Share']].head())

print("\n=== DATA TYPES ===")
print(df[['Total_Revenue', 'Commercial_Revenue', 'Wages_Ratio', 'Wages_Amount', 'Commercial_Share']].dtypes)

print("\n=== SUMMARY STATISTICS ===")
print(df[['Wages_Amount', 'Commercial_Share']].describe())

print("\n=== MISSING VALUES AFTER CONVERSION ===")
print("Rows with missing Wages_Ratio:", df['Wages_Ratio'].isna().sum())
print("Rows with missing Total_Revenue:", df['Total_Revenue'].isna().sum())
print("Rows with missing Commercial_Revenue:", df['Commercial_Revenue'].isna().sum())

## Step 3: Part 1 - 2024 Cross-Sectional Analysis (20 Clubs)

In [ ]:
# Filter for 2024 data
df_2024 = df[df['Year'] == 2024].copy()

# Remove rows with missing Wages_Ratio for visualization
df_2024 = df_2024.dropna(subset=['Wages_Ratio'])

print(f"=== 2024 DATA ===")
print(f"Number of clubs in 2024: {len(df_2024)}")
print(f"Average Wages/Revenue Ratio: {df_2024['Wages_Ratio'].mean():.1f}%")
print(f"UEFA FFP Warning Threshold: 70%")
print(f"Clubs above threshold: {len(df_2024[df_2024['Wages_Ratio'] > 70])}")

if df_2024.empty:
    print("⚠ No data for 2024! Check 'Year' column.")
    print("Available years:", df['Year'].unique())

### Visualization 1: Safety & Compliance (Bar Chart)

**Accounting Insight**: UEFA Financial Fair Play regulations recommend wages should not exceed 70% of revenue.

In [ ]:
# Create bar chart for Wages/Revenue Ratio
if not df_2024.empty:
    # Sort for better visualization
    df_2024_sorted = df_2024.sort_values('Wages_Ratio', ascending=False)
    
    plt.figure(figsize=(14, 8))
    
    # Create bar plot with color coding
    bars = plt.bar(df_2024_sorted['Club'], df_2024_sorted['Wages_Ratio'], 
                   color=['#e74c3c' if x > 70 else '#3498db' for x in df_2024_sorted['Wages_Ratio']])
    
    # Add UEFA FFP threshold line
    plt.axhline(y=70, color='red', linestyle='--', linewidth=2, label='UEFA FFP Warning Threshold (70%)')
    
    # Customize chart
    plt.title('Wages/Revenue Ratio (%) - 2024 Season\nUEFA Financial Fair Play Compliance', fontweight='bold', pad=20)
    plt.xlabel('Club Name', fontweight='bold')
    plt.ylabel('Wages/Revenue Ratio (%)', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.legend(loc='upper right')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                 f'{height:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()

**Business Insight**: Clubs with ratios above 70% (red bars) are operating with poor cost control and may face UEFA sanctions.

### Visualization 2: Scale vs. Cost (Scatter Plot)

**Accounting Insight**: Analyzing the relationship between total revenue and wage expenditure reveals operational efficiency.

In [ ]:
# Create scatter plot with regression line
if not df_2024.empty:
    plt.figure(figsize=(12, 8))
    
    # Remove rows with missing values for regression calculation
    df_2024_clean = df_2024.dropna(subset=['Total_Revenue', 'Wages_Amount'])
    
    # Scatter plot (use original df_2024 for plotting, but regression uses clean data)
    scatter = plt.scatter(df_2024['Total_Revenue'], df_2024['Wages_Amount'], 
                         s=150, alpha=0.7, edgecolors='black', linewidth=0.5)
    
    # Add linear regression line (using clean data)
    if len(df_2024_clean) >= 2:  # Need at least 2 points for regression
        x = df_2024_clean['Total_Revenue']
        y = df_2024_clean['Wages_Amount']
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        plt.plot(x, p(x), "r--", alpha=0.8, linewidth=2, label=f'Linear Trend: y = {z[0]:.2f}x + {z[1]:.2f}')
        
        # Calculate residuals for all data points (including those with missing values will be NaN)
        df_2024['Residual'] = df_2024['Wages_Amount'] - p(df_2024['Total_Revenue'])
        
        # Find and annotate outliers (excluding NaN residuals)
        df_2024_nonnull = df_2024.dropna(subset=['Residual'])
        high_residual = df_2024_nonnull.nlargest(3, 'Residual')  # High wages relative to revenue
        low_residual = df_2024_nonnull.nsmallest(3, 'Residual')   # Low wages relative to revenue
        
        # Annotate high residual clubs (over-spenders)
        for idx, row in high_residual.iterrows():
            plt.annotate(row['Club'], 
                        (row['Total_Revenue'], row['Wages_Amount']),
                        xytext=(10, 10), textcoords='offset points',
                        fontsize=10, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))
        
        # Annotate low residual clubs (under-spenders)
        for idx, row in low_residual.iterrows():
            plt.annotate(row['Club'], 
                        (row['Total_Revenue'], row['Wages_Amount']),
                        xytext=(10, -15), textcoords='offset points',
                        fontsize=10, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7))
    else:
        print("Warning: Not enough valid data points for regression analysis.")
    
    # Customize chart
    plt.title('Scale vs. Cost: Total Revenue vs. Wage Expenditure (2024)', fontweight='bold', pad=20)
    plt.xlabel('Total Revenue (€ millions)', fontweight='bold')
    plt.ylabel('Wages Amount (€ millions)', fontweight='bold')
    if len(df_2024_clean) >= 2:
        plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    # Add correlation coefficient (using clean data)
    if len(df_2024_clean) >= 2:
        correlation = df_2024_clean['Total_Revenue'].corr(df_2024_clean['Wages_Amount'])
        plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                 transform=plt.gca().transAxes, fontsize=12,
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

**Business Insight**: The positive correlation indicates that larger clubs generally pay higher wages. Clubs above the trendline are over-spending relative to revenue scale.

## Step 4: Part 2 - 5-Year Longitudinal Case Study (3 Clubs)

In [ ]:
# Prepare performance data
performance_df_clean = performance_df.copy()
performance_df_clean = performance_df_clean.rename(columns={'Club': 'Club_Perf'})

# Merge with financial data
# We need to match club names - they might differ slightly
print("=== MERGING DATA ===")
print("Clubs in financial data:", df['Club'].unique()[:10])
print("Clubs in performance data:", performance_df_clean['Club_Perf'].unique())

# Simple merge assuming names match exactly
merged_df = pd.merge(df, performance_df_clean, 
                     left_on=['Club', 'Year'], 
                     right_on=['Club_Perf', 'Year'], 
                     how='inner')

print(f"\n✓ Merged data shape: {merged_df.shape}")
print("Clubs in merged data:", merged_df['Club'].unique())
print("\nFirst few rows:")
print(merged_df[['Club', 'Year', 'Total_Revenue', 'Commercial_Revenue', 'Wages_Ratio', 'European Result']].head())

### Visualization 3: The Cost of Success (Multi-line Chart)

**Accounting Insight**: Tracking Wages/Revenue Ratio over time reveals wage inflation trends and financial discipline.

In [ ]:
# Line chart for Wages Ratio trend
if not merged_df.empty:
    plt.figure(figsize=(14, 8))
    
    # Colors for clubs
    club_colors = {
        'Real Madrid': '#004D98',
        'AC Milan': '#FF0000',
        'Paris Saint-Germain': '#004170'
    }
    
    # Plot each club's trend
    for club in merged_df['Club'].unique():
        club_data = merged_df[merged_df['Club'] == club].sort_values('Year')
        color = club_colors.get(club, '#333333')
        
        plt.plot(club_data['Year'], club_data['Wages_Ratio'], 
                 marker='o', markersize=10, linewidth=3, 
                 color=color, label=club)
        
        # Add data labels
        for _, row in club_data.iterrows():
            plt.text(row['Year'], row['Wages_Ratio'] + 1, 
                     f'{row["Wages_Ratio"]:.1f}%', ha='center', va='bottom', fontsize=10)
    
    # Add UEFA FFP threshold line
    plt.axhline(y=70, color='red', linestyle='--', linewidth=2, alpha=0.7, label='UEFA FFP Threshold (70%)')
    
    # Customize chart
    plt.title('The Cost of Success: Wages/Revenue Ratio Trend (2020-2024)', fontweight='bold', pad=20)
    plt.xlabel('Year', fontweight='bold')
    plt.ylabel('Wages/Revenue Ratio (%)', fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

**Business Insight**: Clubs maintaining stable or decreasing ratios demonstrate effective cost control despite revenue growth.

### Visualization 4: Performance vs. Revenue (Dual-Axis Chart)

**Accounting Insight**: This visualization examines the time lag between sporting success and financial rewards.

In [ ]:
# Dual-axis chart for each club
if not merged_df.empty:
    clubs_to_plot = merged_df['Club'].unique()[:3]  # First 3 clubs
    
    for club in clubs_to_plot:
        club_data = merged_df[merged_df['Club'] == club].sort_values('Year')
        
        if len(club_data) > 1:
            fig, ax1 = plt.subplots(figsize=(10, 6))
            
            # Bar chart for total revenue
            ax1.bar(club_data['Year'] - 0.2, club_data['Total_Revenue'], 
                   width=0.4, color='#3498db', alpha=0.7, label='Total Revenue')
            ax1.set_xlabel('Year', fontweight='bold')
            ax1.set_ylabel('Total Revenue (€ millions)', fontweight='bold', color='#3498db')
            ax1.tick_params(axis='y', labelcolor='#3498db')
            
            # Line chart for commercial revenue (second y-axis)
            ax2 = ax1.twinx()
            ax2.plot(club_data['Year'], club_data['Commercial_Revenue'], 
                    marker='s', markersize=10, linewidth=3, 
                    color='#e74c3c', label='Commercial Revenue')
            ax2.set_ylabel('Commercial Revenue (€ millions)', fontweight='bold', color='#e74c3c')
            ax2.tick_params(axis='y', labelcolor='#e74c3c')
            
            # Add European result annotations
            for _, row in club_data.iterrows():
                if pd.notna(row.get('European Result')):
                    result = str(row['European Result'])
                    if result.lower() not in ['nan', 'none', 'n/a', '']:
                        ax2.annotate(result, 
                                   (row['Year'], row['Commercial_Revenue']),
                                   xytext=(0, 15), textcoords='offset points',
                                   ha='center', fontsize=9, fontweight='bold',
                                   bbox=dict(boxstyle="round,pad=0.3", facecolor="gold", alpha=0.8))
            
            plt.title(f'{club}: Revenue Growth & European Performance (2020-2024)', fontweight='bold', pad=20)
            
            # Combine legends
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
            
            plt.tight_layout()
            plt.show()

**Business Insight**: European success typically boosts commercial revenue in subsequent years through increased sponsorship value and merchandise sales.

## Step 5: Export and Conclusion

In [ ]:
# Export transformed data
df.to_csv('financial_data_transformed.csv', index=False, encoding='utf-8-sig')
if not merged_df.empty:
    merged_df.to_csv('merged_data_final.csv', index=False, encoding='utf-8-sig')

print("=== EXPORT COMPLETE ===")
print("✓ financial_data_transformed.csv")
if not merged_df.empty:
    print("✓ merged_data_final.csv")

print("\n=== ANALYSIS COMPLETE ===")
print("All required visualizations have been generated.")
print("Please refer to the charts above for accounting insights.")

## Accounting Recommendations

### Key Findings:
1. **Cost Control Compliance**: Monitor clubs exceeding UEFA's 70% wage/revenue threshold
2. **Revenue Scale Advantage**: Larger clubs achieve economies of scale in wage management
3. **Commercial Diversification**: Top performers show higher commercial revenue shares
4. **Success Lag Effect**: European trophy wins boost commercial revenue 1-2 years later

### Strategic Recommendations:
1. **For Club CFOs**: Implement wage/revenue ratio caps at 60% for compliance buffers
2. **For Investors**: Prioritize clubs with declining wage ratios and rising commercial share
3. **For UEFA Regulators**: Consider multi-year ratio averages for fair assessment
